<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-11-cost-and-performance/lesson-11.5-self-hosting-with-ollama/notebooks/exercises-11.5.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 11.5 — Self-Hosting with Ollama
Netsetos GenAI Course v2.0 | Module 11: Cost & Performance

Setup, quantization tiers, Modelfile, deployment, edge, India MSME use cases.


In [ ]:
print('Module 11: Cost & Performance')
print('Lesson 11.5: Self-Hosting with Ollama')


## Ex 1: Ollama vs vLLM vs llama.cpp


In [ ]:
print('Decision matrix:')
for tool, dx, throughput, best_for in [
    ('Ollama',    'easy (brew/curl + 1 cmd)', 'low (single-proc)', 'dev, laptops, edge, internal'),
    ('vLLM',      'medium (Python + GPU)',    'high (cont. batch)', 'production GPU serving'),
    ('llama.cpp', 'low-level (compile)',     'medium',             'embedded, max portability'),
    ('TGI (HF)',  'medium (Docker)',          'high',               'HF model deployments'),
]:
    print(f'  {tool:12s}: dx={dx:30s} | throughput={throughput:18s} | best: {best_for}')


## Ex 2: Quantization Tiers (Llama-3.1-8B)


In [ ]:
print('Quantization trade-off (Llama-3.1-8B GGUF):')
for quant, size_gb, quality_pct, notes in [
    ('FP16',   16.0, 100, 'baseline, large RAM'),
    ('Q8_0',    8.5,  99, 'near-baseline, halved'),
    ('Q5_K_M',  5.7,  98, 'great quality, smaller'),
    ('Q4_K_M',  4.9,  96, 'production sweet spot'),
    ('Q4_0',    4.5,  94, 'older quant, slightly worse'),
    ('Q3_K_M',  3.6,  90, 'noticeable degradation'),
    ('Q2_K',    2.7,  82, 'not for production'),
]:
    print(f'  {quant:8s} {size_gb:5.1f} GB | quality {quality_pct:>3}% | {notes}')
print()
print('Q4_K_M default. Test on YOUR eval set; some tasks (math, code) are quant-sensitive.')


## Ex 3: Hardware Sizing Table


In [ ]:
print('What hardware runs what (Q4_K_M):')
for hw, ram, runs, throughput in [
    ('M1/M2 Mac 16GB',     16, '8B comfortably',  '~12 tok/s'),
    ('M3 Max 64GB',        64, 'up to 70B',       '~7 tok/s for 70B'),
    ('RTX 4090 24GB',      24, '8B fast, 13B OK', '~80 tok/s for 8B'),
    ('A100 80GB',          80, 'up to 70B + headroom', '~50 tok/s for 70B'),
    ('Raspberry Pi 5 8GB',  8, '7B Q4 slowly',    '~3 tok/s'),
    ('Laptop CPU 16GB',    16, '7B-8B Q4',        '~6 tok/s'),
]:
    print(f'  {hw:24s} | {ram:>2}GB | runs: {runs:18s} | {throughput}')


## Ex 4: Modelfile (Dockerfile for LLMs)


In [ ]:
modelfile = '''\
FROM llama3.1:8b
SYSTEM "You are a Netsetos triage agent. Classify support tickets into category + severity. Be concise."
PARAMETER temperature 0.2
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
PARAMETER stop "</answer>"
'''
print(modelfile)
print('Build: ollama create netsetos-triage -f Modelfile')
print('Run:   ollama run netsetos-triage "my server is down"')
print('Share: ollama push namespace/netsetos-triage')


## Ex 5: OpenAI-Compatible Drop-In


In [ ]:
print('# Drop-in: same code, no API spend')
print()
print('from openai import OpenAI')
print('c = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")')
print('r = c.chat.completions.create(')
print('    model="llama3.1:8b",')
print('    messages=[{"role":"user","content":"explain RAG in 2 sentences"}]')
print(')')
print('print(r.choices[0].message.content)')
print()
print('Same SDK, same code shape, zero API cost. Toggle base_url to switch local <-> cloud.')


## Ex 6: Production Limitations


In [ ]:
print('Ollama production gotchas:')
for issue, impact, mitigation in [
    ('Single-process serving',     'sequential users wait',         'pin 1 model/instance, scale N pods'),
    ('No continuous batching',     'low GPU utilization',           'graduate to vLLM at QPS > 1'),
    ('Cold model load 5-30s',      'first user sees lag',           'pre-warm via /api/embeddings'),
    ('No multi-tenant auth',       'anyone on network can query',   'nginx + token / mTLS in front'),
    ('Memory leaks long-running',  'OOM after days',                'daily restart, watch RSS'),
    ('VRAM contention multi-model','OOM if both loaded',            'pin one model per instance'),
]:
    print(f'  {issue:30s}: {impact:30s} | mitigation: {mitigation}')


## Ex 7: India MSME On-Prem Stack


In [ ]:
print('India MSME on-prem reference stack:')
for component, choice, cost_inr in [
    ('GPU box',           'RTX 4090 24GB workstation',       150_000),
    ('OS',                'Ubuntu 22.04 LTS',                       0),
    ('Inference',         'Ollama + Llama-3.1-8B Q4_K_M',           0),
    ('Reverse proxy',     'nginx with token auth',                  0),
    ('UI',                'open-webui (self-host)',                 0),
    ('Backup model',      'qwen2.5-coder:7b',                       0),
]:
    print(f'  {component:20s}: {choice:36s} | Rs {cost_inr:>8,.0f}')
print()
print(f'Total CAPEX: ~Rs 1,50,000 one-time. OPEX: electricity ~Rs 800/mo.')
print('Serves <100 internal users on Llama-3.1-8B with no API spend.')


---
## Recap
Ollama = dev + edge + internal MSME. 30s install, OpenAI-compatible API, Modelfile for repro.
Q4_K_M is the production sweet spot for 8B (5 GB, 96% quality).
Single-process limits prod scale — graduate to vLLM at sustained QPS > 1.
RTX 4090 box (Rs 1.5L) serves <100 internal users with zero API spend.
